# 第3章　第一个训练循环（线性回归）

把第2章的梯度下降，用到"从数据学一条直线"的问题上。
在这里 **PyTorch 训练的"五步核心"** 就完成了。再大的模型，骨架都一样。

本章目标：能从数据学出 $y = wx + b$ 的 $w,b$，理解 `optimizer` 的作用。

> **本笔记使用方法**
> - 从上到下依次运行单元格（Colab/Jupyter 都是 `Shift + Enter`）。
> - 代码**稍作修改、弄坏再修好**最能进步。每章末尾有练习。
> - 多数章节不需要 GPU。较重的章节（CNN）会说明用法。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 3-1. 造练习数据
准备"真实关系是 $y = 2x + 1$"再加一点噪声的数据。目标就是把它拟合出来。

In [ ]:
import torch
torch.manual_seed(0)                       # 固定随机数以便复现

X = torch.linspace(-3, 3, 100).unsqueeze(1)   # 形状 (100, 1)
true_w, true_b = 2.0, 1.0
y = true_w * X + true_b + 0.5 * torch.randn(X.shape)   # 带噪声

print("X:", X.shape, " y:", y.shape)

## 3-2. 先"手动"训练（彻底理解内部）

- 用 `requires_grad=True` 准备参数 `w, b`。
- **预测（forward）**：`pred = w*X + b`
- **损失（loss）**：预测与真值的偏差。回归用**均方误差 (MSE)** $\frac1N\sum (pred-y)^2$。
- **梯度（backward）** → **更新** → **清零**。

In [ ]:
w = torch.zeros(1, requires_grad=True)   # 从 0 开始
b = torch.zeros(1, requires_grad=True)
lr = 0.05

for epoch in range(100):
    pred = w * X + b                  # ① 预测（forward）
    loss = ((pred - y) ** 2).mean()   # ② 损失（MSE）

    loss.backward()                   # ③ 计算梯度
    with torch.no_grad():
        w -= lr * w.grad              # ④ 更新
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()    # ⑤ 清零

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}: loss={loss.item():.4f}  w={w.item():.3f}  b={b.item():.3f}")

print(f"学到的  w={w.item():.3f} (真值2.0),  b={b.item():.3f} (真值1.0)")

## 3-3. 用 `optimizer` 写同样的事（实战风格）

手动更新部分（`w -= lr*w.grad ...` 和 `zero_()`）由 `torch.optim` 一并完成。
这里登场的就是 **五步核心**：

```
optimizer.zero_grad()   # ① 清零梯度
pred = model(x)         # ② 预测（forward）
loss = criterion(...)   # ③ 损失
loss.backward()         # ④ 梯度
optimizer.step()        # ⑤ 更新
```

`nn.Linear(1, 1)` 是"输入1维→输出1维的直线"，即内部带 `w, b` 的模型。

In [ ]:
import torch.nn as nn

model = nn.Linear(1, 1)                       # 表示 y = w*x + b 的模型
criterion = nn.MSELoss()                      # 损失函数（均方误差）
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)   # 优化算法

for epoch in range(100):
    optimizer.zero_grad()        # ① 清零梯度
    pred = model(X)              # ② 预测
    loss = criterion(pred, y)    # ③ 损失
    loss.backward()              # ④ 梯度
    optimizer.step()             # ⑤ 更新

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}: loss={loss.item():.4f}")

w_learned = model.weight.item()
b_learned = model.bias.item()
print(f"学到的  w={w_learned:.3f} (真值2.0),  b={b_learned:.3f} (真值1.0)")

## 3-4. 可视化结果
确认学到的直线是否穿过数据中间。

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():                 # 只预测，不需要梯度
    pred_line = model(X)

plt.figure(figsize=(6, 4))
plt.scatter(X.numpy(), y.numpy(), s=12, label="data")
plt.plot(X.numpy(), pred_line.numpy(), color="red", linewidth=2, label="learned line")
plt.legend(); plt.title("Linear Regression"); plt.xlabel("x"); plt.ylabel("y")
plt.show()

## 练习 3
1. 把 `lr` 改成 `0.5` 或 `0.001`，观察收敛快慢／发散。
2. 把真实关系改成 `y = -3x + 2`，看能否正确学出。
3. 把 `SGD` 换成 `torch.optim.Adam(model.parameters(), lr=0.1)`，收敛有何变化？
4. 用 `losses.append(loss.item())` 记录损失，再 `plt.plot(losses)` 画学习曲线。

In [ ]:
# 在这里写你自己的代码并运行
